# Phase 34 -- re-measuring q15 and the 8 residual Phase 21 questions under production

Two remaining backlog items, run together since both are "does the current model/prompt
still have this known gap":

1. **q15** -- Phase 32 found this didn't recover to Phase 25's judge score; Phase 33 then
   found the actual mechanism (the same completeness-guard rank issue as mt02) and fixed
   it in `src/calc.py`. This section confirms what q15's answer looks like now.
2. **The 8 residual Phase 21 questions** (q01, q05, q09, q19, q29, q35, q42, q49) -- flagged
   as open since Phase 21, diagnosed by a subagent as generation gaps (content was present
   in retrieved context but never used correctly), never re-measured under the Phase 31
   model switch (Claude Haiku 4.5 + compact prompt) or Phase 28's rule 21/22 additions.

**Method:** no LLM judge -- reuses the exact judge reasons already on file
(`eval/rereview_judge.csv`, Phase 31, `gpt-4o-mini` production answers graded by `gpt-4o`)
as the concrete, specific defect to check for on each question, then generates a fresh
answer with TODAY's production pipeline and reads it directly against that defect. This
is more precise than a fresh judge call for a question-by-question defect check, and
avoids the open problem (no well-calibrated judge exists for Haiku, per Phase 31).
Questions with any sign of inconsistency get multiple trials, per Phase 23a's rule that a
sampling model needs repeated trials before a behavioural claim is trusted.


In [1]:
import sys
import json
from pathlib import Path

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
sys.path.insert(0, str(PROJECT))

from dotenv import load_dotenv
load_dotenv(PROJECT / ".env")

import pandas as pd

from src.retrieval import Retriever
from src.generation import get_production_generator, PRODUCTION_MODEL_ID

EVAL_DIR = PROJECT / "eval"
print("production model:", PRODUCTION_MODEL_ID)

QIDS = ["q01", "q05", "q09", "q15", "q19", "q29", "q35", "q42", "q49"]

eval_v3 = {}
with open(EVAL_DIR / "eval_v3.jsonl", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        if d["question_id"] in QIDS:
            eval_v3[d["question_id"]] = d


production model: anthropic/claude-haiku-4.5


## The known defect per question

Pulled from `eval/rereview_judge.csv`'s `gpt-4o-mini (production)` arm (Phase 31, judged
by `gpt-4o`) -- the most recent full judge pass with recorded reasons. q15's defect is
copied from the same source (used in Phase 32/33's analysis, not re-quoted there).


In [2]:
judge = pd.read_csv(EVAL_DIR / "rereview_judge.csv", encoding="utf-8-sig")
prod = judge[judge.arm == "gpt-4o-mini (production)"].set_index("question_id")

for qid in QIDS:
    r = prod.loc[qid]
    print(f"=== {qid} === faith {r.faithfulness_score} / rel {r.relevancy_score}")
    print("  faith:", r.faithfulness_reason)
    print("  rel:  ", r.relevancy_reason)
    print()


=== q01 === faith 5 / rel 5
  faith: All claims about daily allowance, medical care, pensions, and death benefits directly match the cited Articles 25, 29, and 30 of the Social Security Law.
  rel:   The answer comprehensively covers all work-injury benefits (medical care, temporary disability allowance, disability pensions, death pension) directly addressing what social security covers for a workplace injury.

=== q05 === faith 4 / rel 4
  faith: Most claims match the source documents, but the answer omits the compensation-basis distinction (last wage vs. average of last six months for task workers) from Article 91 and doesn't clarify these are two separate systems (Labor Law vs SSL) that may apply depending on coverage.
  rel:   The answer directly addresses death and permanent disability compensation with figures and calculation basis, but omits mention of Article 91's calculation method and the distinction of which law applies to whom (Article 86), leaving the 'how the amount is wo

## Fresh generations under today's production pipeline

Single-turn, no history (all 9 are single-turn questions in `eval_v3`). q09 and q29 get
extra trials -- q09 because an initial spot-check outside this notebook showed
inconsistent behaviour across repeats, q29 because its defect (omitted heir-type detail)
is a finer-grained coverage check worth confirming isn't a one-off read.


In [3]:
RESULTS_PATH = EVAL_DIR / "residual_and_q15_recheck.csv"
TRIALS = {"q09": 6, "q29": 3}  # default 1 trial for the rest

retriever = Retriever()
gen = get_production_generator()

rows = []
if RESULTS_PATH.exists():
    rows = pd.read_csv(RESULTS_PATH, encoding="utf-8-sig").to_dict("records")
done = {}
for r in rows:
    done[r["question_id"]] = done.get(r["question_id"], 0) + 1

for qid in QIDS:
    d = eval_v3[qid]
    q = d["question_text"]
    n_trials = TRIALS.get(qid, 1)
    chunks = retriever.retrieve(q, top_k=5)
    retrieved = [f"{c['law']}:{c['article_number']}" for c in chunks]
    have = done.get(qid, 0)
    for trial in range(have, n_trials):
        res = gen.generate(q, chunks)
        rows.append({
            "question_id": qid, "trial": trial, "question": q,
            "gold_articles": d["answer_article_numbers"], "retrieved": retrieved,
            "answer": res["answer"],
        })
        pd.DataFrame(rows).to_csv(RESULTS_PATH, index=False, encoding="utf-8-sig")
        print(f"  {qid} / trial {trial} done (retrieved: {retrieved})")

results_df = pd.DataFrame(rows)
print(f"\n{len(results_df)} rows total")



16 rows total


In [4]:
for qid in QIDS:
    sub = results_df[results_df.question_id == qid]
    print("=" * 90)
    print(qid, "--", eval_v3[qid]["question_text"])
    print("gold:", eval_v3[qid]["answer_article_numbers"], "| retrieved:", sub.iloc[0]["retrieved"])
    print()
    for _, r in sub.iterrows():
        print(f"--- trial {r.trial} ---")
        print(r.answer)
        print()


q01 -- I was injured while at work. What does social security actually cover for me?
gold: ['25'] | retrieved: ['labor96:86', 'labor96:90', 'ssl:29', 'ssl:25', 'ssl:30', 'ssl:4']

--- trial 0 ---
Social Security covers work injuries for you if you are subject to the Social Security Law (generally, workers subject to the Labour Law who have completed 16 years of age and earn at least the minimum wage). If you don't fall under Social Security coverage, the Labour Law's injury provisions apply instead.

**If you are covered by Social Security, Work-Injury Insurance includes:**

**Medical care:**  
Social Security Law, Article 25(a) — the medical care required by your health condition.

**Temporary inability to work (while injured and unable to work):**  
Social Security Law, Article 29(a) — a daily allowance of 75% of your daily wage (based on the wage used to calculate your contributions on the date of injury), for days you are under treatment at an approved centre or at home per Medical

## Verdict

**7 of 9 questions read as fixed under today's production pipeline (Haiku 4.5, compact prompt, Phase 28's rule 21/22, Phase 33's calc fix).** 2 are still genuinely open, with repeated-trial evidence.

**Fixed:**
- **q01** -- still comprehensive and correct (faith/rel 5/5 originally, unchanged).
- **q05** -- the two prior omissions are both gone: the answer now leads with "depends on whether the person is covered by Social Security Law or only by Labour Law -- two separate systems," and separately states Article 91's calculation basis (last wage, or average of the last six months for task-based workers). This is Phase 28's rule 21/Article 86 precondition mechanism visibly working on the exact question it was built for.
- **q15** -- not just fixed, dramatically expanded. The prior complaint (missing age-based reduction table and dependent increases) is now covered in detail: the answer gives all three of Article 64's pension formulas, each with its own age-based reduction table (both sexes) and dependent-increase percentages, and is honest where the corpus itself is incomplete ("Table 5 ... not fully provided in these documents"). This is Phase 33's rank-exception fix (compute_gender_conditions now firing on this single-article question) plus the model doing a genuinely thorough coverage job on top of it.
- **q19** -- now correctly splits work-injury disability (Article 36(a): every 6 months, within 2 years) from natural disability (Article 67(e): any time within 2 years) as two clearly separate cases, instead of conflating them.
- **q35** -- still fully correct (5/5 originally, unchanged).
- **q42** -- correctly states the 10-week total is split before/after birth with only 6 weeks guaranteed after, instead of implying all 10 weeks are post-birth.
- **q49** -- the previously-omitted Ministry-of-Labor two-week notification requirement is now stated up front and repeated in an obligations section.

**Still open, real gaps (repeated-trial evidence, not a single unlucky read):**
- **q09** ("can I work while receiving a work-injury disability pension") -- only **1 of 6** fresh trials cleanly separates the actually-applicable Article 40 (work-injury disability pension -- yes, may combine) from Article 85(e)/(f) (a *different* pension, "Natural Disability Pension"). The other **5 of 6** mislabel 85(e)/(f) as governing "Work-Related Disability" -- i.e. the same *category* of misapplication the original Phase 21/25/31 complaint identified, just landing on different sub-paragraphs. Two of those five are seriously wrong: trial 5 never mentions Article 40 at all and flatly answers "No, you cannot work" for a total work-injury disability pension, which is the opposite of the governing rule; trial 0 states "cannot combine" and then separately quotes Article 40 saying the insured "may combine," an unresolved internal contradiction. **Not fixed** -- this is a live, still-present, majority-of-the-time defect under the current production model.
- **q29** ("what are the conditions for a deceased husband's heirs to get a pension") -- across all 3 trials here, Article 81's per-heir-type conditions are at best a bare list of eligible categories ("widow, daughters, sons, parents") and at worst omitted entirely (trial 2 doesn't mention Article 81 at all). None of the 3 trials surface the actual *conditions* Article 81 attaches per type -- the age-23 cutoff with a total-disability exception for sons, the rule that a daughter/sister's share stops on marriage and resumes on divorce/widowhood, or the husband's-share condition (total disability + no equivalent independent income, made up to the difference). **Not fixed** -- the original complaint stands.

**Read together with Phase 33:** three separate mechanisms are now confirmed working as designed on real questions (Article 86 precondition for q05, the rank-exception gender-block fix for q15, the natural-language distinction logic for q19/q42/q49). The two still-open items (q09, q29) are both cases where the retrieved article text itself is unusually dense with adjacent-but-different sub-clauses (Article 85's several disability-pension variants; Article 81's several heir categories each with distinct conditions) -- the same shape of failure Phase 21 originally flagged as "generation gap, not retrieval gap," now confirmed to persist under a different model and a rewritten prompt. Not something a prompt reword is likely to reliably fix on its own, per this project's repeated finding (Phase 24d, 25a) that prompt-only fixes narrow a failure rate rather than eliminate it -- if q09 specifically is worth fixing to a hard guarantee, the precedent here is Phase 28's Article 86 precondition mechanism: a structural, code-level fix rather than another wording pass.


## Deep dive: fixing q09 and q29

Reading the actual retrieved article text for both explains *why* the model keeps
getting them wrong, not just that it does.

**q09 -- Article 40 vs Article 85.** Article 40 is a single sentence that directly
answers the question: a Work Injury Disability Pension may be combined with wages,
no condition stated. Article 85 gets retrieved right alongside it (both are about
combining a Social Security pension with something else) and turns out to contain
two separate, genuine traps in its own text: paragraph (b) mentions "work-related
disability pensioner" by name -- but that paragraph is about combining with ANOTHER
pension, not wages, so it's a keyword match on the wrong axis. Paragraphs (e) and (f)
ARE about combining a disability pension with wages -- but they say "Natural
Disability Pension", a different, non-work-injury insurance branch. Neither paragraph
governs the question asked, and the model has been citing one or the other as if it
did, across every measurement since Phase 21. **Fix: a pure disambiguation note**,
`disability_pension_scope_note` in `src/calc.py`, fires whenever both Article 40 and
Article 85 (SSL) are retrieved together and states plainly which of Article 85's
paragraphs govern which (different) situation. No arithmetic, no precondition to ask
the person -- unlike everything else in `calc.py` this is not a computed figure, just
a scope clarification injected into context exactly like the other calculator blocks.

**q29 -- Article 81's conditions live one level too deep for the coverage rule.**
Article 81 was already being retrieved (often ranked #1), so this was never a
retrieval gap. Reading it directly: its substantive content -- the four heir
categories and each one's specific condition (sons: under 23, with a total-disability
exception and biennial re-exam; widow/daughters/sisters: share suspended on marriage,
resumed on divorce or widowhood; husband: total disability plus no equivalent income,
made up to the difference; parents: unconditional) -- sits as NUMBERED sub-items
nested inside a single lettered paragraph, (a). `SYSTEM_PROMPT` rule 5 said to cover
every *lettered paragraph*, and never mentioned numbered items nested inside one --
so an article like 64 (many top-level lettered paragraphs, confirmed thoroughly
covered in the q15 recheck above) got the full treatment, while an article like 81
(one lettered paragraph holding four numbered conditions) did not. **Fix: broadened
rule 5** to explicitly name nested numbered points, not just lettered paragraphs.


In [5]:
POSTFIX_PATH = EVAL_DIR / "residual_q09_q29_postfix.csv"
POSTFIX_TRIALS = {"q09": 6, "q29": 6}

postfix_qs = {
    "q09": eval_v3["q09"]["question_text"],
    "q29": eval_v3["q29"]["question_text"],
}

postfix_rows = []
if POSTFIX_PATH.exists():
    postfix_rows = pd.read_csv(POSTFIX_PATH, encoding="utf-8-sig").to_dict("records")
pdone = {}
for r in postfix_rows:
    pdone[r["question_id"]] = pdone.get(r["question_id"], 0) + 1

for qid, q in postfix_qs.items():
    chunks = retriever.retrieve(q, top_k=5)
    have = pdone.get(qid, 0)
    for trial in range(have, POSTFIX_TRIALS[qid]):
        res = gen.generate(q, chunks)
        postfix_rows.append({"question_id": qid, "trial": trial, "answer": res["answer"]})
        pd.DataFrame(postfix_rows).to_csv(POSTFIX_PATH, index=False, encoding="utf-8-sig")
        print(f"  {qid} / trial {trial} done")

postfix_df = pd.DataFrame(postfix_rows)
for qid in postfix_qs:
    print("=" * 90, qid)
    for _, r in postfix_df[postfix_df.question_id == qid].iterrows():
        print(f"--- trial {r.trial} ---")
        print(r.answer)
        print()


  q09 / trial 0 done


  q09 / trial 1 done


  q09 / trial 2 done


  q09 / trial 3 done


  q09 / trial 4 done


...

  q29 / trial 1 done


  q29 / trial 2 done


  q29 / trial 3 done


  q29 / trial 4 done


  q29 / trial 5 done
========================================================================================== q09
--- trial 0 ---
Yes. Under the Social Security Law, Article 40, an insured person may combine earnings from a job or occupation with a Work Injury Disability Pension.

This applies without further condition being stated in the law — you do not need to meet waiting periods, employer restrictions, or pension reduction thresholds that apply to other types of pensions.

General information only, not legal advice; consult a licensed lawyer about your own case.

--- trial 1 ---
Yes. Under the Social Security Law, Article 40, you may combine what you earn from a job or occupation with your Work Injury Disability Pension.

This is general information, not legal advice; consult a licensed lawyer about your own case.

--- trial 2 ---
Yes. **Social Security Law, Article 40** states: "The insured may combine between what he/she earns from a job or an occupation and his/her Work Injur

## Deep-dive verdict

**q09 -- FIXED, adopted.** `disability_pension_scope_note()` in `src/calc.py`,
wired into `src/generation.py`. Measured **12/12 clean across two separate runs**
(6 trials before this notebook was formalized, 6 more in the reproducible run above)
-- every trial now gives a short, correct "Yes, Article 40" answer with no mention of
Article 85 at all, up from 1/6 before the fix. This is the strongest result of any
fix in this project's history on a pure generation-side (non-arithmetic) defect --
the note removes the ambiguity at the source rather than asking the model to reason
its way past two genuinely similarly-worded, adjacent-but-different paragraphs.
**Adopted.**

**q29 -- IMPROVED, adopted, but NOT reliably fixed -- disclosed as a partial result.**
The broadened rule 5 in `SYSTEM_PROMPT` (`src/generation.py`). Measured across 12
trials total (6 ad-hoc before formalizing this section, 6 more in the reproducible
run above): **roughly 3-4 of 12 (~25-30%) now give the actual per-heir-type
conditions** (the age-23/disability exception for sons, the marriage-suspends-share
condition for the widow/daughters/sisters), up from **0 of 9 total across every prior
measurement** (Phase 31's original run, Phase 34's first recheck, this notebook's
baseline). The other ~70% still either omit Article 81 entirely or give only a bare
category list. **This matches the project's established pattern (Phase 24d, 25a):
a prompt-only fix narrows a failure rate, it does not eliminate it** -- especially
here, where the fix is a *rule interpretation* change (the model must still notice
and choose to apply the deeper coverage requirement) rather than removing an actual
source of ambiguity the way q09's note does. **Adopted anyway** -- it is a real,
positive, measured improvement (0% to ~30%) with no observed regression on unrelated
questions (q01/q35/q42 spot-checked unaffected), and the broadened wording is
generically correct (nested numbered conditions inside a lettered paragraph is a real
gap the rule should have covered regardless of q29). **If q29-level reliability is
wanted, the applicable precedent is `compute_gender_conditions`/`compute_end_of_service`
-- a deterministic extractor reading Article 81's four heir categories and their
conditions directly from text, the same pattern already used for contributions,
end-of-service, gender-differentiated pensions, overtime, and leave accrual.** Not
built here -- it's a real engineering step up from a scope-note (the article has four
distinct condition shapes to parse, not one number pair), and is a decision left for
the user rather than taken unilaterally.

**Net effect on the original 9-question backlog: 8 of 9 now reliably fixed (q01, q05,
q09, q15, q19, q35, q42, q49); 1 of 9 (q29) measurably improved but still open.**
